<a id="top"></a>
# Marketing bancario: validazione del primo modello

La fase di preparazione ha prodotto un dataset numerico, senza valori mancanti, adatto a un modello di classificazione binaria che preveda la sottoscrizione del deposito. 
Qui l’attenzione si sposta sulla **validazione**: quanto si può considerare affidabile il modello su clienti mai visti prima e in che misura risponde alla richiesta del business.

<a id="indice"></a>
## Indice

- [Obiettivo del modulo e dataset di partenza](#obiettivo-modulo)
- [Ricostruzione rapida del dataset per il modello](#dataset-modello)
- [Train/test split e class imbalance](#train-test)
- [Metriche per un problema sbilanciato](#metriche-sbilanciato)
- [Curva ROC e AUROC](#roc-auroc)
- [Curva precision-recall e AUPR](#pr-curve)
- [Matrice di confusione e costi di errore](#confusione)
- [Dalle metriche alle leve di business](#bridge-explain)
- [Permutation Feature Importance](#pfi)
- [SHAP summary plot](#shap)
- [Sintesi operativa](#sintesi)


<a id="obiettivo-modulo"></a>
## Obiettivo del modulo e dataset di partenza

Il team marketing ha chiesto un **PoC** che risponda a due domande essenziali:

1. *È realistico prevedere quali clienti sottoscriveranno il deposito?*
2. *Se la risposta è sì, quali fattori incidono di più sulla probabilità di sottoscrizione?*

Per la prima domanda serve una **misura quantitativa** delle prestazioni del modello su dati non utilizzati in addestramento.
Per la seconda serve una **lettura strutturata delle feature**, che consenta di trasformare il modello in indicazioni operative.

Tutto il lavoro parte dal dataset costruito nel modulo di preparazione, dove:
- il target `y` è binario (1 = sottoscrive il deposito, 0 = non sottoscrive);
- le feature sono numeriche, prive di valori mancanti, con `duration` esclusa dalle variabili di input.

<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">
<strong>Nota importante</strong><br>
Senza una metrica chiara non si può dire se il PoC abbia raggiunto il livello richiesto dal business.
Si può migliorare solo ciò che viene misurato in modo coerente e ripetibile.
</div>


<a id="dataset-modello"></a>
## Ricostruzione rapida del dataset per il modello

Si ricrea in forma compatta lo stesso dataset numerico del modulo precedente:

- normalizzazione di `y` a 0/1;
- gestione di `default`, `housing`, `loan` e costruzione di `has_any_loan`;
- trattamento di `pdays` e creazione di `was_previously_contacted`;
- encoding di `education` come variabile ordinale;
- one-hot encoding delle variabili categoriali nominali;
- imputazione multivariata con `IterativeImputer`;
- identificazione di `X_imputed` (feature) e `y_target` (target binario).


In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np

# Caricamento del dataset Bank Marketing (id=222 nel repository UCI)
bank_marketing = fetch_ucirepo(id=222)

X_raw = bank_marketing.data.features
y_raw = bank_marketing.data.targets

df = X_raw.copy()
if isinstance(y_raw, pd.DataFrame):
    if y_raw.shape[1] == 1:
        df["y"] = y_raw.iloc[:, 0]
    else:
        for col in y_raw.columns:
            df[col] = y_raw[col]
else:
    df["y"] = y_raw

print("Shape feature raw:", X_raw.shape)
print("Shape target raw:", y_raw.shape)
df.head()


In [ ]:
# Preparazione compatta delle variabili per il modello
df_prep = df.copy()

# 1. Target binario y: yes/no -> 1/0
if "y" in df_prep.columns:
    df_prep["y"] = df_prep["y"].map({"yes": 1, "no": 0}).astype("int8")

# 2. Colonne yes/no con possibile valore 'unknown'
bool_like_cols = [col for col in ["default", "housing", "loan"] if col in df_prep.columns]

for col in bool_like_cols:
    df_prep[col] = df_prep[col].replace("unknown", np.nan)
    df_prep[col] = df_prep[col].map({"yes": 1, "no": 0})

# 3. Feature di sintesi: presenza di qualunque mutuo/prestito
if all(col in df_prep.columns for col in ["housing", "loan"]):
    both_unknown = df_prep[["housing", "loan"]].isna().all(axis=1)
    any_yes = (df_prep[["housing", "loan"]] == 1).any(axis=1)
    df_prep["has_any_loan"] = np.where(
        any_yes,
        1,
        np.where(both_unknown, np.nan, 0)
    )

# 4. Gestione di pdays e creazione di was_previously_contacted
if "pdays" in df_prep.columns:
    df_prep["was_previously_contacted"] = (df_prep["pdays"] != -1).astype("int8")
    df_prep["pdays"] = df_prep["pdays"].replace(-1, np.nan)

df_prep.head()


In [ ]:
# Encoding delle variabili categoriali
df_enc = df_prep.copy()

# Variabile education come ordinale
if "education" in df_enc.columns:
    education_mapping = {
        "primary": 0,
        "secondary": 1,
        "tertiary": 2,
        "unknown": np.nan
    }
    df_enc["education"] = df_enc["education"].map(education_mapping)

# Variabili nominali da espandere con one-hot encoding
possible_nominal = ["job", "marital", "contact", "month", "poutcome"]
nominal_cols = [col for col in possible_nominal if col in df_enc.columns]

df_enc = pd.get_dummies(
    df_enc,
    columns=nominal_cols,
    drop_first=True
)

print("Forma del dataset dopo encoding:", df_enc.shape)
df_enc.head()


In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer

# Separiamo target e feature candidate, escludendo duration dagli input
y_target = df_enc["y"]
feature_cols = [col for col in df_enc.columns if col not in ["y", "duration"]]
X_features = df_enc[feature_cols]

imputer = IterativeImputer(random_state=42)
X_imputed_array = imputer.fit_transform(X_features)
X_imputed = pd.DataFrame(X_imputed_array, columns=feature_cols, index=df_enc.index)

print("Numero di feature modellabili:", len(feature_cols))
print("Valori mancanti residui:", int(X_imputed.isna().sum().sum()))
X_imputed.head()


<a id="train-test"></a>
## Train/test split e class imbalance

Per valutare la **capacità di generalizzazione** del modello si separa il dataset in due parti:

- un **train set** usato per addestrare il modello;
- un **test set** usato solo per valutare il comportamento su clienti non visti.

Il dataset è sbilanciato: i clienti che sottoscrivono il deposito (`y = 1`) sono una minoranza.
Prima di addestrare il modello conviene quantificare questo squilibrio e fissare una baseline ingenua.


In [ ]:
from sklearn.model_selection import train_test_split

# Distribuzione del target complessiva
target_counts = y_target.value_counts(normalize=True).sort_index()
print("Distribuzione relativa del target (0/1):")
print(target_counts)

# Train/test split con stratificazione sul target
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed,
    y_target,
    test_size=0.2,
    random_state=42,
    stratify=y_target
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


<div style="background-color:#fff3cd; padding:10px; border-left:4px solid #c69500;">
<strong>Attenzione</strong><br>
Il test set deve restare "pulito": va usato una sola volta per stimare le prestazioni del modello.
Se iperparametri e scelte di feature vengono continuamente modificati guardando al test, 
la stima finale risulta troppo ottimistica.
</div>


<a id="metriche-sbilanciato"></a>
## Metriche per un problema sbilanciato

In un dataset sbilanciato l’**accuratezza** può essere fuorviante.
Un classificatore che predice sempre la classe maggioritaria ottiene un’accuracy elevata,
ma non risolve il problema del business perché non individua quasi mai i clienti che sottoscrivono.

Servono metriche che tengano conto della separazione tra le due classi:

- **AUROC** (Area Under ROC Curve): misura la capacità del modello di ordinare correttamente i positivi rispetto ai negativi;
- **AUPR** (Area Under Precision-Recall Curve): mette l’accento sulle prestazioni nella classe positiva, utile quando i positivi sono pochi.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Modello di riferimento: Random Forest sugli input imputati (non scalati)
rf_clf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
)

rf_clf.fit(X_train, y_train)

# Baseline ingenua: predire sempre la classe più frequente del train
majority_class = y_train.value_counts().idxmax()
y_pred_baseline = np.full_like(y_test, fill_value=majority_class)

baseline_acc = accuracy_score(y_test, y_pred_baseline)
model_acc = accuracy_score(y_test, rf_clf.predict(X_test))

print(f"Accuracy baseline (sempre classe {majority_class}): {baseline_acc:.3f}")
print(f"Accuracy modello Random Forest: {model_acc:.3f}")


<a id="roc-auroc"></a>
## Curva ROC e AUROC

La **curva ROC** (Receiver Operating Characteristic) rappresenta il compromesso tra:

- *True Positive Rate* (sensibilità): quota di positivi correttamente identificati;
- *False Positive Rate*: quota di negativi classificati erroneamente come positivi.

L’**AUROC** è l’area sotto questa curva. Valori vicini a 0.5 indicano un modello poco utile,
valori più alti indicano una migliore separazione tra clienti che sottoscrivono e clienti che non sottoscrivono.


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# Probabilità previste per la classe positiva
y_proba_test = rf_clf.predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_proba_test)
auroc = roc_auc_score(y_test, y_proba_test)

print(f"AUROC: {auroc:.3f}")

plt.figure()
plt.plot(fpr, tpr, label=f"ROC (AUROC = {auroc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Modello casuale")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC - Bank Marketing")
plt.legend()
plt.grid(True)
plt.show()


<a id="pr-curve"></a>
## Curva precision-recall e AUPR

Quando la classe positiva è rara, la **curva precision-recall** spesso racconta meglio il comportamento del modello.

- La **precision** indica quale quota dei clienti classificati come positivi sottoscrive davvero;
- il **recall** coincide con la sensibilità, cioè la quota di clienti che sottoscrive e viene correttamente intercettata.

L’**AUPR** sintetizza l’andamento della curva: valori più alti indicano un modello capace di mantenere una buona precision
anche a livelli di recall interessanti per la campagna.


In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

precision, recall, pr_thresholds = precision_recall_curve(y_test, y_proba_test)
aupr = average_precision_score(y_test, y_proba_test)

print(f"AUPR (Average Precision): {aupr:.3f}")

plt.figure()
plt.plot(recall, precision, label=f"PR (AUPR = {aupr:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Curva precision-recall - Bank Marketing")
plt.legend()
plt.grid(True)
plt.show()


<a id="confusione"></a>
## Matrice di confusione e costi di errore

Le curve ROC e precision-recall riassumono bene il comportamento del modello su tutte le soglie possibili.
Per passare a decisioni concrete serve anche una lettura in termini di **conteggi di casi corretti e sbagliati**.

La **matrice di confusione** mette in evidenza:

- *True Negative* (TN): clienti correttamente classificati come non sottoscrittori;
- *False Positive* (FP): clienti contattati e classificati come interessati ma che non sottoscrivono;
- *False Negative* (FN): clienti che avrebbero sottoscritto ma che il modello non segnala;
- *True Positive* (TP): clienti correttamente identificati come sottoscrittori.

Nel contesto della campagna:
- gli FP rappresentano chiamate "sprecate" (tempo del call center, possibile fastidio per il cliente);
- gli FN rappresentano opportunità perse di sottoscrizione.


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred_test = rf_clf.predict(X_test)
cm = confusion_matrix(y_test, y_pred_test)

tn, fp, fn, tp = cm.ravel()
print("Matrice di confusione (righe = vero, colonne = predetto):")
print(cm)
print()
print(f"TN: {tn}, FP: {fp}, FN: {fn}, TP: {tp}")

print("\nReport di classificazione:")
print(classification_report(y_test, y_pred_test, digits=3))


<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">
<strong>Esercizio</strong><br>
Varia il parametro <code>n_estimators</code> della Random Forest e osserva come cambiano AUROC, AUPR e matrice di confusione.
Prova a spiegare quali differenze sono solo dovute al rumore statistico e quali invece sembrano tendenze strutturali.
</div>


<a id="bridge-explain"></a>
## Dalle metriche alle leve di business

Le metriche sul test set danno una prima risposta alla domanda *“il modello funziona abbastanza bene?”*.
Per trasformare il PoC in uno strumento utile al marketing manca però un tassello: capire **da cosa dipendono**
le probabilità di sottoscrizione.

Il team chiede di identificare:
- quali feature hanno l’impatto maggiore sulle previsioni;
- in che direzione agiscono sui singoli clienti.

Si introducono quindi due strumenti complementari:

- la **Permutation Feature Importance** (PFI), che misura quanto peggiorano le prestazioni quando una feature viene permutata;
- i **valori SHAP**, che decomponogno la previsione di ogni cliente come somma di contributi delle singole feature.


<a id="pfi"></a>
## Permutation Feature Importance

La **Permutation Feature Importance** valuta l’importanza di una feature misurando quanto peggiora una metrica
di interesse quando i valori di quella feature vengono permutati in modo casuale.

In scikit-learn è implementata in `sklearn.inspection.permutation_importance`.
Documentazione: https://scikit-learn.org/stable/modules/permutation_importance.html


In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    rf_clf,
    X_test,
    y_test,
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

importances_mean = result.importances_mean
importances_std = result.importances_std

feature_importance = pd.DataFrame({
    "feature": X_imputed.columns,
    "importance_mean": importances_mean,
    "importance_std": importances_std,
}).sort_values("importance_mean", ascending=False)

feature_importance.head(10)


In [ ]:
# Visualizzazione delle prime 15 feature per importanza permutazionale
top_n = 15
fi_top = feature_importance.head(top_n)

plt.figure(figsize=(8, 6))
plt.barh(fi_top["feature"][::-1], fi_top["importance_mean"][::-1])
plt.xlabel("Variazione media della metrica")
plt.title("Permutation Feature Importance (test set)")
plt.tight_layout()
plt.show()


<a id="shap"></a>
## SHAP summary plot

I **valori SHAP** (SHapley Additive exPlanations) forniscono una decomposizione additiva della previsione di un modello
in contributi delle singole feature. Per ogni cliente è possibile vedere quali variabili spingono la probabilità di
sottoscrizione verso l’alto o verso il basso.

Per modelli ad alberi come la Random Forest esiste un `TreeExplainer` dedicato.
Documentazione: https://shap.readthedocs.io/en/latest/


In [ ]:
# !pip install shap  # se la libreria non è già presente nell'ambiente
import shap

# SHAP può essere costoso su dataset grandi; si usa un campione del test set per il grafico
sample_size = min(1000, X_test.shape[0])
X_test_sample = X_test.sample(sample_size, random_state=42)

explainer = shap.TreeExplainer(rf_clf)
shap_values = explainer.shap_values(X_test_sample)

# Per un problema binario shap_values è una lista con un array per classe
shap_values_positive = shap_values[1]

shap.summary_plot(shap_values_positive, X_test_sample, show=True)


<a id="sintesi"></a>
## Sintesi operativa

Il percorso di validazione ha portato a tre risultati principali:

1. **Misura della qualità predittiva**: AUROC e AUPR sul test set offrono una base numerica per dire se il PoC
   è in linea con le aspettative del business, tenendo conto dello sbilanciamento del dataset.
2. **Lettura degli errori**: la matrice di confusione consente di quantificare falsi positivi e falsi negativi,
   collegando direttamente le prestazioni del modello a costi e opportunità della campagna.
3. **Fattori chiave individuati**: PFI e SHAP summary plot mostrano quali feature guidano le previsioni e in che direzione,
   preparando il terreno per un confronto con il team marketing sulle leve più efficaci.

Il PoC non si esaurisce in un numero di metrica, ma in un quadro coerente di prestazioni ed elementi interpretativi
su cui costruire eventuali iterazioni successive.
